# 00a · 랜덤 워크로 시작하기 (Starting with the Random Walk)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §1.1 (Example 1.1 gambler's ruin), §1.8 (exit distributions) · Ross 12e §2.2.2 (binomial), §4.5.1 (gambler's ruin) · G&S 4e §3.9 (simple random walk), §3.10 (counting sample paths) · 정리 D 인용 출처(읽기 목록 밖): G&S 4e §5.10, Ross 12e §2.7 (CLT) |
| 선수 노트북 | 없음 (학부 확률: 이항분포, 분산의 덧셈, CLT, 기하급수) |
| 예상 소요 | 90분 |
| 상태 | draft |

이 노트북의 단 하나의 아이디어: **랜덤 워크 $S_n$은 $\pm1$ 걸음의 누적합이고, 그 퍼짐은 $n$이 아니라 $\sqrt n$ 규모다.** 그리고 이 과정 전체의 작업 루프(예측 → 시뮬레이션 → 비교표 → assert → 기록)를 여기서 처음 한 바퀴 돌린다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.stats import binom, norm
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import random_walk_paths, gamblers_ruin_prob, gamblers_ruin_duration
from spkit.plots import plot_paths, plot_hist_vs_pmf, plot_hist_vs_pdf

SEED, rng = rng_for("00a")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

첫 노트북이므로 학부 확률에서 가져올 도구를 확인한다. 책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (학부 확률) $X \sim \mathrm{Bin}(n,p)$의 평균과 분산은?

<details><summary>정답</summary>

$E[X]=np$, $\mathrm{Var}(X)=np(1-p)$. 독립 베르누이 $n$개의 합이므로 평균·분산이 모두 더해진다.

</details>

**Q2.** (학부 확률) $X_i$가 i.i.d., 평균 $\mu$, 분산 $\sigma^2$일 때 $S_n=\sum_{i\le n}X_i$의 분산과 중심극한정리(CLT)의 진술은?

<details><summary>정답</summary>

$\mathrm{Var}(S_n)=n\sigma^2$ (독립이라 공분산 항이 0). CLT: $(S_n-n\mu)/(\sigma\sqrt n)\Rightarrow N(0,1)$.

</details>

**Q3.** (학부 확률) $P(X=1)=P(X=-1)=1/2$인 $X$의 평균·분산은? 그러면 $S_n$의 평균·분산은?

<details><summary>정답</summary>

$E[X]=0$, $\mathrm{Var}(X)=E[X^2]=1$. 따라서 $E[S_n]=0$, $\mathrm{Var}(S_n)=n$: 퍼짐의 크기는 $n$이 아니라 $\sqrt n$.

</details>

**Q4 (연결 고리).** (학부 수학) $1+r+r^2+\cdots+r^{n-1}$의 닫힌 형식은? $r=1$이면?

<details><summary>정답</summary>

$r\ne1$: $(1-r^n)/(1-r)$. $r=1$: $n$. — 파산 공식에서 $p=1/2$ ($r=q/p=1$)를 따로 다뤄야 하는 이유가 바로 이것이다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **단순 랜덤 워크(simple random walk).** $X_1,X_2,\dots$ i.i.d., $P(X_i=1)=p$, $P(X_i=-1)=q=1-p$, $S_0=0$, $S_n=\sum_{i=1}^n X_i$. $p=1/2$이면 **대칭(symmetric)** 이라 부른다.
- **$S_n$의 분포.** $K_n=(S_n+n)/2$는 위로 간 걸음 수이므로 $K_n\sim\mathrm{Bin}(n,p)$. 따라서
  $$P(S_n=k)=\binom{n}{(n+k)/2}p^{(n+k)/2}q^{(n-k)/2}\quad(k\equiv n \pmod 2),$$
  그 외의 $k$에서는 0. $E[S_n]=n(p-q)$, $\mathrm{Var}(S_n)=4npq$.
- **도박꾼의 파산(gambler's ruin).** 시작 자본 $i$, 목표 $N$ ($0<i<N$). 위치 $i+S_n$이 $0$ 또는 $N$에 처음 닿는 시각 $T=\min\{n\ge0: i+S_n\in\{0,N\}\}$. 승리확률 $h(i)=P_i(i+S_T=N)$, 기대 지속시간 $D(i)=E_i[T]$.
- **정규 근사(de Moivre–Laplace).** $S_n\approx N(n(p-q),\,4npq)$. **연속성 수정(continuity correction):** $P(S_n\ge k)\approx P\big(Z\ge (k-1-n(p-q))/\sqrt{4npq}\big)$. $S_n$의 격자 간격이 2이므로 수정량은 $\pm1$이다($\pm1/2$가 아님).

### 2.2 정리 A — 파산 확률 (gambler's ruin probability; Durrett 3e §1.8, Ross 12e §4.5.1, G&S 4e §3.9)

> $0<p<1$, $q=1-p$, $0\le i\le N$일 때 $h(i)=P_i(\text{N에 먼저 도달})$는
> $$h(i)=\begin{cases}\dfrac{i}{N}, & p=\tfrac12,\\[2ex] \dfrac{1-(q/p)^i}{1-(q/p)^N}, & p\ne\tfrac12.\end{cases}$$

**가정이 왜 필요한가**
- **걸음의 독립성.** 첫걸음 분석 $h(i)=p\,h(i+1)+q\,h(i-1)$은 첫걸음 이후의 미래가 현재 위치만으로 결정된다는 데 기댄다. 걸음이 과거에 의존하면(E3의 persistent walk) 이 방정식 자체가 틀린다.
- **동일분포(시간에 따라 $p$가 변하지 않음).** 차분방정식의 계수가 상수여야 특성방정식 $p r^2-r+q=0$으로 풀 수 있다.
- **$0<p<1$.** $p\in\{0,1\}$이면 자명하고, 증명의 "$T<\infty$ a.s." 단계가 $p^N>0$을 쓴다.
- **$p\ne1/2$와 $p=1/2$의 분리.** 특성근이 $1$과 $q/p$인데 $p=1/2$이면 중근이라 일반해가 기하급수 꼴이 아니라 1차식 $A+Bi$가 된다. 일반 공식에 $p=1/2$를 넣으면 $0/0$.

<details><summary>증명</summary>

**(1) $T<\infty$ a.s., 그리고 $E_i[T]<\infty$.** 어느 위치 $x\in\{1,\dots,N-1\}$에서든 연속 $N$번 위로 가면(확률 $p^N>0$) 위치가 $N$ 이상이 되어 게임이 끝난다. 따라서 길이 $N$의 블록마다 독립적으로 "끝날 확률 $\ge p^N$"이 있으므로
$$P_i(T>kN)\le(1-p^N)^k\xrightarrow{k\to\infty}0,$$
즉 $T<\infty$ a.s. 같은 부등식에서 $E_i[T]=\sum_{n\ge0}P_i(T>n)\le N\sum_{k\ge0}(1-p^N)^k=N/p^N<\infty$.

**(2) 첫걸음 조건화.** $0<i<N$에서 첫걸음 $X_1$로 조건화하면, 첫걸음 뒤의 워크는 위치 $i\pm1$에서 새로 시작하는 같은 종류의 워크이므로(독립·동일분포)
$$h(i)=p\,h(i+1)+q\,h(i-1),\qquad h(0)=0,\quad h(N)=1.$$
$h(i)=(p+q)h(i)$로 쓰고 정리하면 $p\,(h(i+1)-h(i))=q\,(h(i)-h(i-1))$. 차분 $d_i=h(i)-h(i-1)$ ($i=1,\dots,N$)에 대해 $d_{i+1}=(q/p)\,d_i$, 따라서 $d_i=(q/p)^{i-1}d_1$.

**(3) 합산.** $h(0)=0$이므로 $h(i)=\sum_{j=1}^{i}d_j=d_1\sum_{j=0}^{i-1}(q/p)^j$ (Recall Q4의 기하급수).
- $p\ne1/2$: $h(i)=d_1\dfrac{1-(q/p)^i}{1-(q/p)}$. $h(N)=1$에서 $d_1=\dfrac{1-(q/p)}{1-(q/p)^N}$이므로 $h(i)=\dfrac{1-(q/p)^i}{1-(q/p)^N}$.
- $p=1/2$: $q/p=1$이라 $h(i)=i\,d_1$, $h(N)=1$에서 $d_1=1/N$, 즉 $h(i)=i/N$. $\square$

**(1)이 필요한 이유.** 주의: (2)–(3)의 경계값 문제 $h(i)=p\,h(i+1)+q\,h(i-1)$, $h(0)=0$, $h(N)=1$ 자체는 (1) 없이도 유일하게 풀린다 — 유한한 $N$에서는 일반해 $A+B(q/p)^i$(또는 $A+Bi$)의 상수 두 개를 경계조건 두 개가 정하고, 첫걸음 항등식도 마르코프 성질(독립·동일분포)만으로 성립하며 $T$의 유한성을 쓰지 않는다. (1)의 역할은 다른 곳에 있다.
- **'패배'가 '승리'의 여사건임을 보장한다.** $T<\infty$ a.s.이므로 $P_i(\text{0에 먼저 도달})=1-h(i)$이다. 게임이 영원히 안 끝나는 사건이 양의 확률을 가지면 $h(i)+P_i(\text{0에 도달})<1$이 되어 파산 확률은 $1-h(i)$보다 작아지고, '승리확률 $h$'만 알아서는 파산 확률을 말할 수 없다.
- **$E_i[T]<\infty$를 준다.** 정리 C와 E2의 $D(i)=1+p\,D(i+1)+q\,D(i-1)$이 유한한 수들 사이의 정당한 항등식이 되려면($\infty-\infty$ 방지) 이것이 필요하다.

(대조: 반직선 $N=\infty$처럼 상태공간이 무한하면 경계조건이 하나만 남아 해가 유일하지 않고, 최소 비음해를 골라야 한다 — 이후 무한 상태공간의 사슬에서 다시 만난다.) `spkit.markov.gamblers_ruin_prob`는 정확히 (3)의 두 식으로 분기한다(4절에서 확인).

</details>

### 2.3 정리 B — $S_n$의 분포와 홀짝성 (G&S 4e §3.9, Ross 12e §2.2.2)

> $K_n=(S_n+n)/2\sim\mathrm{Bin}(n,p)$이고 $S_n\equiv n\pmod 2$. 특히 $p=1/2$이면 $P(S_{2m}=0)=\binom{2m}{m}4^{-m}\sim\dfrac{1}{\sqrt{\pi m}}$.

**가정이 왜 필요한가**
- **i.i.d. 걸음.** 위로 간 걸음 수가 이항분포가 되려면 각 걸음이 독립 베르누이여야 한다.
- **걸음 크기가 정확히 $\pm1$.** $S_n=2K_n-n$이라는 1:1 대응이 홀짝성의 근원이다. 걸음 크기가 $\{-1,0,1\}$이면 홀짝 제약이 사라진다.

<details><summary>증명</summary>

위로 간 걸음 수를 $K_n=\#\{i\le n: X_i=+1\}$, 아래로 간 걸음 수를 $n-K_n$이라 하면 $S_n=K_n-(n-K_n)=2K_n-n$. $K_n$은 독립 베르누이($p$) $n$개의 합이므로 $\mathrm{Bin}(n,p)$이고, $S_n=2K_n-n\equiv n\pmod2$. 역으로 $K_n=(S_n+n)/2$이므로 $P(S_n=k)=P(K_n=(n+k)/2)$이고, $(n+k)/2$가 정수가 아니면 확률 0. $E[S_n]=2np-n=n(p-q)$, $\mathrm{Var}(S_n)=4\,\mathrm{Var}(K_n)=4npq$. $\square$

$\binom{2m}{m}4^{-m}\sim(\pi m)^{-1/2}$는 스털링 공식의 결과이며 01x에서 다룬다(여기서는 인용). $m=50$: 정확값 $0.0796$, 근사값 $1/\sqrt{50\pi}=0.0798$.

</details>

### 2.4 정리 C — 기대 지속시간 (expected duration; Ross 12e §4.5.1, G&S 4e §3.9)

> $p=1/2$: $D(i)=i(N-i)$. $\quad p\ne1/2$: $D(i)=\dfrac{i}{q-p}-\dfrac{N}{q-p}\cdot\dfrac{1-(q/p)^i}{1-(q/p)^N}$.

**가정이 왜 필요한가**
- 정리 A와 같은 독립·동일분포 가정 위에서 $D(i)=1+p\,D(i+1)+q\,D(i-1)$이 성립한다(첫걸음에 1을 쓰고, 그 뒤는 새 위치에서의 기대 지속시간).
- $E_i[T]<\infty$ (정리 A 증명의 (1))가 있어야 조건화한 기댓값을 마음 놓고 쓸 수 있다($\infty-\infty$ 방지).

**증명 스케치.** 비동차 차분방정식 $D(i)=1+p\,D(i+1)+q\,D(i-1)$, $D(0)=D(N)=0$. 특수해 하나: $p=1/2$이면 $-i^2$, $p\ne1/2$이면 $i/(q-p)$ (대입해 보면 상수항 1이 정확히 상쇄된다). 일반해 = 특수해 + 동차해(정리 A의 $A+B(q/p)^i$ 또는 $A+Bi$)이고 경계조건 두 개로 $A,B$를 정한다. 완전한 유도는 **E2**에서 학습자가 직접 한다.

### 2.5 정리 D — de Moivre–Laplace / CLT (인용: G&S 4e §5.10, Ross 12e §2.7)

> $\dfrac{S_n-n(p-q)}{\sqrt{4npq}}\Rightarrow N(0,1)$.

**가정이 왜 필요한가.** 유한 분산과 i.i.d. — 00b의 E3(무한 분산 걸음)에서 깨지는 모습을 본다. 여기서는 증명 대신 그림(4.2절 오른쪽 패널)과 꼬리확률 비교(4.3절)로만 확인한다.

### 2.6 직관

걸음 $n$번 뒤 위치의 전형적 크기는 $n$이 아니라 $\sqrt n$이다(**확산적 퍼짐**): $n=1000$이면 $\pm30$ 근처가 보통이고 $\pm100$은 드물다(3 시그마 밖).

파산 문제에서는 **작은 불리함이 치명적**이다: $N=20$, $i=10$에서 공정하면 승리확률 $1/2$이지만 $p=0.45$면 약 $0.12$로 떨어진다. $(q/p)^i$가 $i$에 대해 지수적으로 커지기 때문이다 — 카지노가 큰 이유는 확률의 5%가 아니라 지수의 5%다.

정상적인 히스토그램이 pmf의 절반으로 보이는 것은 버그가 아니라 **홀짝성**이다: `density=True` 막대의 높이는 (칸 안의 확률질량)/(칸 폭)인데, $S_n$의 원자(atom)는 간격 2로 놓여 있다. 그래서 원자 하나를 담은 폭 $w$의 칸은 높이 $\mathrm{pmf}/w$가 된다 — 폭 2 칸(예: `bins=40`, 범위 $[-40,40]$)이면 pmf의 **절반**, 폭 1 칸이면 pmf와 같지만 홀수 칸이 모두 비어 정규 pdf($\approx\mathrm{pmf}/2$)의 **두 배** 높이의 빗살(comb)이 된다.

### 2.7 함정(traps)

1. **홀짝 함정.** 폭 2 칸 히스토그램(`density=True`, 짝수 경계 — `bins=40` 같은 '자연스러운' 선택) 위에 pmf를 겹치면 막대가 pmf의 1/2 높이로 보인다; 폭 1 정수 칸이면 pmf와는 맞지만 정규 pdf의 2배 높이 빗살이 된다. 원인은 원자 간격 2. 해결: (a) $K_n=(S_n+n)/2$로 바꿔 $\mathrm{Bin}(n,1/2)$와 비교(원자 간격 1), 또는 (b) 칸 경계를 **홀수**($-101,-99,\dots,101$)에 두어 폭 2 칸 사용(그러면 density 막대 높이 = pmf/2 = 정규 pdf와 바로 비교 가능; 짝수 경계면 원자가 칸 경계 위에 놓여 어느 칸에 들어가는지가 히스토그램의 반열림 규칙 $[a,b)$에 달리고, 마지막 칸은 닫힌 구간이라 원자 두 개를 담는다 — 예: `bins=40` on $[-40,40]$의 마지막 칸 $[38,40]$은 38과 40을 함께 담아 다른 칸보다 높다).
2. **$p=1/2$를 일반 공식에 대입하면 $0/0$.** 극한 $r\to1$을 취하거나 $i/N$을 따로 써야 한다. `gamblers_ruin_prob`도 분기한다.
3. **미완 경로를 '파산'으로 세지 말 것.** 정해진 걸음 수(예: 1000)만 돌리고 아직 안 끝난 경로를 패배로 세면 승리확률이 편향된다. 모든 경로가 흡수될 때까지 돌리고(active 마스크), 남은 경로 수가 0임을 `assert`한다.
4. **$E|S_n|\ne\sqrt n$.** $E|S_n|\approx\sqrt{2n/\pi}\approx0.80\sqrt n$ ($n=100$: 정확히 $7.9589$; 대칭 워크에서는 $E|S_{2m}|=2m\,P(S_{2m}=0)$이라는 항등식이 있다).
5. **연속성 수정 없는 정규 근사.** $P(|S_{100}|\ge20)$의 정확값 $0.0569$에 대해 수정 없는 근사는 $0.0455$, 수정한 근사는 $0.0574$. 격자 간격이 2이므로 수정량은 $\pm1$이다.
6. **전역 상태 난수 API 금지.** `np.random.seed`, `np.random.choice` 등은 구조 검사에서 실패한다. 항상 `SEED, rng = rng_for("00a")`의 `rng`를 넘긴다.

**쓰기 전 길잡이 질문** (다음 셀에 답하세요):
- 첫걸음 분석에서 차분방정식 $h(i)=p\,h(i+1)+q\,h(i-1)$이 나오는 한 문장짜리 이유는?
- 왜 $p=1/2$에서는 답이 $i/N$(1차식)이고 $p\ne1/2$에서는 기하급수 꼴인가? (Recall Q4와 연결해서)

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

## 3. Predict — 코드를 돌리기 전에 예측

아래 셀의 `None`을 숫자로 바꿔 보세요. 정확할 필요는 없습니다 — 자릿수와 방향이 맞는지가 목적입니다. 예측을 먼저 적어 두고 4절을 실행하세요. 5절의 비교표에서 이 값이 '내 예측' 열에 들어갑니다.

In [ ]:
predictions = {
    # 대칭 랜덤 워크에서 P(S_100 = 0)는? (힌트: C(100,50)/2^100; 0.5? 0.1? 0.01?)
    "p_zero_100": None,
    # P(|S_100| >= 20)는? (표준편차가 10이므로 '2 시그마 밖'이다. 정규 근사값 0.0455와 같을까?)
    "p_far_100": None,
    # N = 20, 시작 i = 10, p = 1/2일 때 N에 먼저 도달할 확률은?
    "ruin_fair": None,
    # 같은 설정에서 p = 0.45 (아주 살짝 불리)일 때 승리확률은? 0.4쯤일까?
    "ruin_unfair": None,
    # N = 20, i = 10, p = 1/2일 때 게임이 끝날 때까지의 기대 걸음 수 E[T]는?
    "duration_fair": None,
    # p = 0.45일 때 E[T]는 100보다 클까 작을까? 숫자로.
    "duration_unfair": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 벡터화된 걸음 생성과 $\sqrt n$ 포락선

경로를 파이썬 `for` 루프로 한 걸음씩 만들지 않는다. $\pm1$ 걸음을 `(n_paths, n_steps)` 행렬로 **한 번에** 뽑고 `cumsum(axis=1)`으로 경로를 만든다 — 이 습관이 이 과정 전체의 코드 스타일이다. `spkit.markov.random_walk_paths`가 같은 일을 한다.

In [ ]:
n_paths, n_steps = 10, 1000
steps = rng.choice([-1, 1], size=(n_paths, n_steps))                   # (10, 1000) 개의 ±1
S_manual = np.concatenate([np.zeros((n_paths, 1), dtype=int), steps.cumsum(axis=1)], axis=1)
S_lib = random_walk_paths(n_paths, n_steps, rng)                       # 같은 구성, spkit 버전

assert S_manual.shape == S_lib.shape == (n_paths, n_steps + 1)
assert (S_manual[:, 0] == 0).all() and (S_lib[:, 0] == 0).all()
assert np.abs(np.diff(S_lib, axis=1)).max() == 1                        # 걸음 크기 정확히 1
print("shape:", S_lib.shape, "| final positions:", S_lib[:, -1])
print(f"typical |S_n| ~ sqrt(n) = {np.sqrt(n_steps):.1f};  observed mean |S_n| = {np.abs(S_lib[:, -1]).mean():.1f}")

In [ ]:
t = np.arange(n_steps + 1)
fig, ax = plt.subplots()
plot_paths(t, S_lib, ax=ax, n_show=10, alpha=0.7)
for c, ls in [(1, "--"), (2, ":")]:
    ax.plot(t, c * np.sqrt(t), "k", ls=ls, lw=1.2, label=f"+/- {c} sqrt(n)")
    ax.plot(t, -c * np.sqrt(t), "k", ls=ls, lw=1.2)
ax.set_xlabel("n"); ax.set_ylabel("S_n")
ax.set_title("Fig 1: Ten symmetric simple random walks, n = 1000 steps")
ax.legend(loc="upper left"); plt.show()

경로 10개 중 대부분이 $\pm\sqrt n$ 안팎에서 움직이고, $\pm2\sqrt n$ 포락선을 넘는 시각은 드물다(정리 D: $S_n/\sqrt n\approx N(0,1)$이므로 임의의 시각에 밖에 있을 확률 약 5%).

### 4.2 $S_{100}$의 분포와 홀짝 함정 (정리 B)

$S_{100}$을 `N_PATHS`개 뽑는다. 먼저 홀짝성을 `assert`로 확인하고, 세 가지 방식으로 히스토그램을 그린다: (a) '자연스러운' `bins=40` (범위 $[-40,40]$, 즉 짝수 경계의 폭 2 칸) — 일부러 함정을 본다, (b) $K=(S+100)/2$ vs $\mathrm{Bin}(100,1/2)$, (c) 홀수 경계의 폭 2 칸 vs $N(0,100)$ 밀도.

In [ ]:
n = 100
paths = random_walk_paths(N_PATHS, n, rng)
S = paths[:, -1]
assert np.all(S % 2 == 0), "S_100 must be even (parity)"
K = (S + n) // 2

k_even = np.arange(-n, n + 1, 2)                       # S_100 의 지지집합 (짝수)
pmf_S = binom.pmf((k_even + n) // 2, n, 0.5)            # P(S_100 = k) = P(K = (k+n)/2)
k_bin = np.arange(0, n + 1)
edges_odd = np.arange(-n - 1, n + 3, 2)                  # 폭 2 칸, 경계가 홀수: 원자가 칸 한가운데
x = np.linspace(-40, 40, 400)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(S, bins=40, range=(-40, 40), density=True, alpha=0.5, label="simulation, bins=40")
axes[0].stem(k_even, pmf_S, linefmt="C1-", markerfmt="C1o", basefmt=" ", label="exact pmf of S_100")
axes[0].set_xlim(-40, 40); axes[0].set_title("(a) naive width-2 bins: bars at HALF pmf height"); axes[0].legend()
plot_hist_vs_pmf(K, k_bin, binom.pmf(k_bin, n, 0.5), ax=axes[1], label_pmf="Binomial(100, 1/2) pmf")
axes[1].set_xlim(30, 70); axes[1].set_title("(b) K = (S_100 + 100)/2 vs Binomial")
plot_hist_vs_pdf(S, x, norm.pdf(x, 0, np.sqrt(n)), ax=axes[2], bins=edges_odd, label_pdf="N(0, 100) pdf")
axes[2].set_xlim(-40, 40); axes[2].set_title("(c) odd-edge width-2 bins vs N(0, 100)")
for ax in axes:
    ax.set_xlabel("value"); ax.set_ylabel("density")
fig.suptitle(f"Fig 2: Distribution of S_100 from {N_PATHS} paths"); plt.tight_layout(); plt.show()

(a)에서 막대가 pmf 점의 정확히 절반 높이인 것은 난수 버그가 아니다: 폭 2 칸 하나에 원자가 하나씩만 들어가므로 `density=True` 높이 = pmf/2 이다. (b)처럼 $K$로 바꾸면 원자 간격이 1이 되어 `plot_hist_vs_pmf`의 폭 1 칸과 맞고, (c)처럼 폭 2 칸을 **정규 pdf**와 비교하면 pmf/2 ≈ pdf 라서 맞는다. 다음 셀에서 폭 1 칸의 '두 배 빗살'도 숫자로 확인한다. (c)의 정규 곡선은 막대와 잘 맞지만, 꼬리에서 얼마나 다른지는 4.3절에서 본다.

In [ ]:
h1, _ = np.histogram(S, bins=np.arange(-40.5, 41.5, 1.0), density=True)   # 폭 1, 정수 중심
h2, _ = np.histogram(S, bins=np.arange(-40, 42, 2.0), density=True)        # 폭 2, 짝수 경계 (= 패널 a)
h3, _ = np.histogram(S, bins=np.arange(-41, 43, 2.0), density=True)        # 폭 2, 홀수 경계 (= 패널 c)
p0, pdf0 = float(binom.pmf(n // 2, n, 0.5)), float(norm.pdf(0, 0, np.sqrt(n)))
print(f"pmf(0) = {p0:.4f},  N(0,100) pdf(0) = {pdf0:.4f} = pmf/2")
print(f"width-1 bar at 0      : {h1[40]:.4f}  (= pmf, but = 2 x pdf; the bar at 1 is {h1[41]:.4f})")
print(f"width-2 even-edge [0,2): {h2[20]:.4f}  (= pmf/2 -> looks 'half height' against the pmf)")
print(f"width-2 odd-edge [-1,1): {h3[20]:.4f}  (= pmf/2 = pdf -> compare with the normal curve)")

### 4.3 세 수치: $P(S_{100}=0)$, $P(|S_{100}|\ge20)$, $E[S_{100}^2]$

같은 `S`로 세 값을 추정하고, 정확한 이항 계산과 정규 근사(연속성 수정 유/무)를 나란히 놓는다.

In [ ]:
est_zero = mc_proportion(S == 0)
est_far = mc_proportion(np.abs(S) >= 20)
est_sq = mc_estimate(S.astype(float) ** 2)

exact_zero = float(binom.pmf(n // 2, n, 0.5))            # C(100,50)/2^100
exact_far = float(2 * binom.sf(59, n, 0.5))              # |S|>=20  <=>  K>=60 or K<=40 (대칭)
exact_sq = float(n)                                      # Var(S_n) = n, E[S_n] = 0
sd = np.sqrt(n)
approx_far = 2 * norm.sf(20 / sd)                        # 연속성 수정 없음: P(|Z| >= 2)
approx_far_cc = 2 * norm.sf((20 - 1) / sd)               # 연속성 수정 (격자 간격 2 -> ±1): P(|Z| >= 1.9)
approx_zero = 1 / np.sqrt(np.pi * n / 2)                 # 정리 B의 점근식 1/sqrt(pi m), m = 50

print(f"P(S_100 = 0)      exact {exact_zero:.5f} | asymptotic 1/sqrt(pi m) {approx_zero:.5f} | MC {est_zero}")
print(f"P(|S_100| >= 20)  exact {exact_far:.5f} | normal {approx_far:.5f} | normal+cc {approx_far_cc:.5f} | MC {est_far}")
print(f"E[S_100^2]        exact {exact_sq:.1f} | MC {est_sq}")
print(f"E|S_100|          exact {n * exact_zero:.4f} (= n P(S_n=0)) | sqrt(2n/pi) {np.sqrt(2 * n / np.pi):.4f} | MC {mc_estimate(np.abs(S))}")

수정 없는 정규 근사(0.0455)는 정확값(0.0569)에서 MC의 SE보다 훨씬 멀리 떨어져 있고, 연속성 수정(0.0574)이 그 오차의 대부분을 없앤다. 비교표에는 정확값만 넣는다.

### 4.4 도박꾼의 파산 시뮬레이션 (정리 A, C)

고정 길이 배열 대신, **모든 경로가 흡수될 때까지** 시간 루프를 돈다. 아직 살아 있는 경로의 인덱스 `active`만 갱신하므로 경로가 줄어들수록 빨라진다. 끝났을 때 `active.size == 0`을 `assert`한다(함정 3). 01a/01c에서 같은 문제를 `simulate_chain`/`absorption`으로 다시 푼다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def simulate_ruin(i, N, p, n_paths, rng, max_steps=100_000):
    """자본 i 에서 시작해 {0, N} 에 흡수될 때까지의 (won, T). 모든 경로가 끝날 때까지 돈다."""
    pos = np.full(n_paths, i, dtype=int)
    won = np.zeros(n_paths, dtype=bool)
    T = np.zeros(n_paths, dtype=int)
    active = np.arange(n_paths)
    for t in range(1, max_steps + 1):
        pos[active] += np.where(rng.random(active.size) < p, 1, -1)
        done = (pos[active] == 0) | (pos[active] == N)
        idx = active[done]
        T[idx] = t
        won[idx] = pos[idx] == N
        active = active[~done]
        if active.size == 0:
            break
    assert active.size == 0, f"{active.size} paths not absorbed after {max_steps} steps"
    return won, T

In [ ]:
i0, N0 = 10, 20
N_RUIN = scale(50_000)        # 흡수까지만 돌리므로 싸다 (full 약 0.3 s); p=0.45 의 상대 SE 를 1.2% 로
won_fair, T_fair = simulate_ruin(i0, N0, 0.5, N_RUIN, rng)
won_unfair, T_unfair = simulate_ruin(i0, N0, 0.45, N_RUIN, rng)
est_win_fair, est_win_unfair = mc_proportion(won_fair), mc_proportion(won_unfair)
est_T_fair, est_T_unfair = mc_estimate(T_fair), mc_estimate(T_unfair)
D_fair, D_unfair = gamblers_ruin_duration(i0, N0, 0.5), gamblers_ruin_duration(i0, N0, 0.45)
print(f"p=0.50: P(win) {est_win_fair} | E[T] {est_T_fair} | max T {T_fair.max()}")
print(f"p=0.45: P(win) {est_win_unfair} | E[T] {est_T_unfair} | max T {T_unfair.max()}")

fig, ax = plt.subplots()
bins = np.arange(0, max(T_fair.max(), T_unfair.max()) + 20, 10)
ax.hist(T_fair, bins=bins, density=True, alpha=0.5, label="p = 0.5")
ax.hist(T_unfair, bins=bins, density=True, alpha=0.5, label="p = 0.45")
ax.axvline(D_fair, color="C0", ls="--", label=f"exact E[T] = {D_fair:.1f} (p = 0.5)")
ax.axvline(D_unfair, color="C1", ls="--", label=f"exact E[T] = {D_unfair:.1f} (p = 0.45)")
ax.set_xlim(0, 500); ax.set_xlabel("absorption time T"); ax.set_ylabel("density")
ax.set_title("Fig 3: Absorption time of the gambler's ruin, i = 10, N = 20")
ax.legend(); plt.show()

$T$의 분포는 오른쪽으로 길게 늘어져 있고(최대값이 평균의 몇 배), $T$는 반드시 짝수(10에서 0 또는 20까지 거리 10은 짝수)다. 불리한 게임($p=0.45$)은 더 **빨리** 끝난다: 평균 76걸음 — 0 쪽으로 밀리는 표류(drift)가 있기 때문이다.

### 4.5 시작점 $i$에 대한 승리확률 곡선 (정리 A)

$i=2,4,\dots,18$마다 짧은 시뮬레이션으로 승리 비율을 얻고, 정리 A의 곡선과 겹친다. 오차막대는 `mc_proportion`의 $1.96\,\mathrm{se}$.

In [ ]:
n_sweep = scale(4000)
i_grid = np.arange(2, 19, 2)
i_fine = np.arange(0, N0 + 1)
fig, ax = plt.subplots()
for p, c in [(0.5, "C0"), (0.45, "C1")]:
    est = [mc_proportion(simulate_ruin(i, N0, p, n_sweep, rng)[0]) for i in i_grid]
    ax.errorbar(i_grid, [e.mean for e in est], yerr=[1.96 * e.se for e in est], fmt="o", color=c,
                capsize=3, label=f"MC, p = {p} (95% error bars)")
    ax.plot(i_fine, [gamblers_ruin_prob(i, N0, p) for i in i_fine], "-", color=c, alpha=0.7,
            label=f"exact, p = {p}" + (" (line i/N)" if p == 0.5 else ""))
ax.set_xlabel("starting point i"); ax.set_ylabel("P_i(reach 20 before 0)")
ax.set_title("Fig 4: Gambler's ruin win probability vs starting point, N = 20")
ax.legend(); plt.show()

$p=0.5$에서는 직선 $i/N$, $p=0.45$에서는 아래로 처진 곡선 $(1-(q/p)^i)/(1-(q/p)^N)$이다. $i=10$에서 두 곡선의 차이(0.5 vs 0.12)가 "작은 불리함의 지수적 벌"이다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
h_fair, h_unfair = gamblers_ruin_prob(i0, N0, 0.5), gamblers_ruin_prob(i0, N0, 0.45)
r = 0.55 / 0.45                                             # q/p at p = 0.45
np.testing.assert_allclose(h_unfair, (1 - r**i0) / (1 - r**N0))          # 정리 A의 식 그대로인지 확인
np.testing.assert_allclose(D_unfair, i0 / 0.1 - (N0 / 0.1) * h_unfair)   # 정리 C의 식 그대로인지 확인

rows = [
    {"name": "P(S_100 = 0)", "predicted": predictions["p_zero_100"], "estimate": est_zero, "exact": exact_zero},
    {"name": "P(|S_100| >= 20)", "predicted": predictions["p_far_100"], "estimate": est_far, "exact": exact_far},
    {"name": "E[S_100^2]", "predicted": None, "estimate": est_sq, "exact": exact_sq},
    {"name": "P_10(hit 20 before 0), p = 0.5", "predicted": predictions["ruin_fair"], "estimate": est_win_fair, "exact": h_fair},
    {"name": "P_10(hit 20 before 0), p = 0.45", "predicted": predictions["ruin_unfair"], "estimate": est_win_unfair, "exact": h_unfair},
    {"name": "E_10[T], p = 0.5", "predicted": predictions["duration_fair"], "estimate": est_T_fair, "exact": D_fair},
    {"name": "E_10[T], p = 0.45", "predicted": predictions["duration_unfair"], "estimate": est_T_unfair, "exact": D_unfair},
]
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **P(S_100 = 0)**: 어긋나면 `random_walk_paths`의 걸음이 대칭이 아니거나 $\binom{100}{50}2^{-100}$의 지수를 잘못 센 것이다.
- **P(|S_100| ≥ 20)**: 어긋나면 `binom.sf(59, …)`의 경계(≥60인지 >60인지)를 잘못 잡은 것이다. 정규 근사값(0.0455)을 넣으면 여기서 'investigate'가 뜬다.
- **E[S_100^2]**: 어긋나면 분산의 덧셈($\mathrm{Var}(S_n)=n$)이 깨진 것, 즉 걸음이 독립이 아니다.
- **P_10(hit 20), p = 0.5 / 0.45**: 어긋나면 미완 경로를 패배로 셌거나(함정 3), `gamblers_ruin_prob`의 $p=1/2$ 분기가 잘못됐다.
- **E_10[T], p = 0.5 / 0.45**: 어긋나면 `simulate_ruin`이 $T$를 한 걸음 어긋나게 기록했거나(off-by-one), 정리 C의 특수해 부호가 틀린 것이다.

**첫 mistakes.md 항목.** 표의 '내 예측' 중 닫힌 형식과 3 SE 이상 어긋난 항목 하나를 골라 `log/mistakes.md` 표에 한 줄(날짜 | 00a | 내 예측 | 진실 | 왜 틀렸나)을 추가하세요. 하나도 없으면 가장 자신 없던 항목의 '왜'를 적으세요. 이것이 이 과정의 첫 기록입니다.

In [ ]:
assert_close(est_zero, exact_zero, k=4, name="P(S_100=0)")
assert_close(est_far, exact_far, k=4, name="P(|S_100|>=20)")
assert_close(est_sq, exact_sq, k=4, name="E[S_100^2]")
assert_close(est_win_fair, h_fair, k=4, name="ruin p=0.5")
assert_close(est_win_unfair, h_unfair, k=4, name="ruin p=0.45")
assert_close(est_T_fair, D_fair, k=4, name="duration p=0.5")
assert_close(est_T_unfair, D_unfair, k=4, name="duration p=0.45")
print("all checks passed")

## 6. 연습문제

### E1 계산

대칭 랜덤 워크 $n=6$에서 손으로 구하라: (a) $P(S_6=0)$, (b) $P(S_6=1)$, (c) $P(S_6=2)$, (d) $E[S_6^2]$, (e) $E|S_6|$. 그다음 $N=4$, $i=2$, $p=0.4$의 파산 문제에서 승리확률 $h(2)$를 정리 A의 공식으로 계산하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$K_6=(S_6+6)/2\sim\mathrm{Bin}(6,1/2)$, 분모는 $2^6=64$.
(a) $P(S_6=0)=P(K_6=3)=\binom63/64=20/64=0.3125$.
(b) $S_6$는 짝수만 취하므로(정리 B) $P(S_6=1)=0$.
(c) $P(S_6=2)=P(K_6=4)=\binom64/64=15/64=0.234375$.
(d) $E[S_6^2]=\mathrm{Var}(S_6)=n=6$.
(e) 값 $\pm2$ (각 $15/64$), $\pm4$ (각 $6/64$), $\pm6$ (각 $1/64$): $E|S_6|=2\,(2\cdot15+4\cdot6+6\cdot1)/64=120/64=1.875$. ($\sqrt6\approx2.45$보다 작다 — 함정 4.)

파산: $q/p=0.6/0.4=1.5$, $h(2)=\dfrac{1-1.5^2}{1-1.5^4}=\dfrac{-1.25}{-4.0625}=0.307692$.

검산 코드:
```python
k = np.arange(7); s6 = 2 * k - 6; pm = binom.pmf(k, 6, 0.5)
print(pm[3], pm[4], (s6**2 * pm).sum(), (np.abs(s6) * pm).sum())   # 0.3125 0.234375 6.0 1.875
print(gamblers_ruin_prob(2, 4, 0.4))                                   # 0.3076923
```

</details>

### E2 유도 후 시뮬레이션 검증

$p=1/2$에서 기대 지속시간이 $D(i)=1+\tfrac12D(i+1)+\tfrac12D(i-1)$, $D(0)=D(N)=0$을 만족함을 첫걸음 분석으로 쓰고, 해가 $D(i)=i(N-i)$임을 유도하라(특수해 $-i^2$ + 동차해 $A+Bi$). 이어서 $N=20$, $i=5$에서 `simulate_ruin`으로 $E[T]$를 추정해 $75$와 4 SE 안에서 맞는지 확인하고, $p=0.45$, $i=5$에서도 정리 C의 공식(`gamblers_ruin_duration`)과 대조하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**첫걸음 분석.** $0<i<N$에서 첫걸음에 1을 쓰고, 그 뒤 위치 $i\pm1$에서 새로 시작하므로(독립·동일분포, $E_i[T]<\infty$) $D(i)=1+\tfrac12D(i+1)+\tfrac12D(i-1)$, 경계 $D(0)=D(N)=0$.

**특수해.** $D(i)=-i^2$을 대입: $1+\tfrac12[-(i+1)^2-(i-1)^2]=1-\tfrac12(2i^2+2)=-i^2$. 성립.

**일반해.** 동차식 $D(i)=\tfrac12D(i+1)+\tfrac12D(i-1)$의 해는 1차식 $A+Bi$(중근, 정리 A의 $p=1/2$ 경우)이므로 $D(i)=-i^2+A+Bi$. $D(0)=0\Rightarrow A=0$, $D(N)=0\Rightarrow B=N$. 따라서 $D(i)=i(N-i)$. $D(5)=5\cdot15=75$.

$p=0.45$, $i=5$: 정리 C에서 $D(5)=\dfrac{5}{0.1}-\dfrac{20}{0.1}\cdot h(5)$, $h(5)=\dfrac{1-r^5}{1-r^{20}}=0.031792$ ($r=11/9$), $D(5)=50-200\cdot0.031792=43.6417$.

```python
won5, T5 = simulate_ruin(5, 20, 0.5, N_PATHS, rng)
assert_close(mc_estimate(T5), 75.0, k=4, name="E_5[T], p=0.5")        # 약 75 ± 0.6 (full)
won5u, T5u = simulate_ruin(5, 20, 0.45, N_PATHS, rng)
D5u, h5u = gamblers_ruin_duration(5, 20, 0.45), gamblers_ruin_prob(5, 20, 0.45)
print(mc_estimate(T5u), D5u, "|", mc_proportion(won5u), h5u)          # 43.64,  0.0318
assert_close(mc_estimate(T5u), D5u, k=4, name="E_5[T], p=0.45")
```

</details>

### E3 가정을 깨보기

걸음의 독립성을 깨자: **지속성(persistent) 랜덤 워크** — $X_1\sim$ 대칭 $\pm1$, $k\ge1$에 대해 $X_{k+1}=X_k$ (확률 $\rho=0.8$) 또는 $-X_k$ (확률 $0.2$). 걸음 하나하나는 여전히 대칭 $\pm1$이다. $N=20$, $i=5$에서 `simulate_ruin`을 변형(직전 걸음 배열 `last`를 유지)해 승리확률을 추정하고 $i/N=0.25$와 비교하라. $i=10$에서는 어떤가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

정리 A의 첫걸음 분석은 $h$가 **위치만의 함수**라는 전제를 쓰는데, 지속성 걸음에서는 (위치, 직전 방향)이 상태다. 시뮬레이션은 $0.25$가 아니라 약 $0.283$을 준다(20000 경로에서 $\pm0.003$). $i=10$은 대칭성으로 여전히 $0.5$ — 공식이 우연히 맞는 경우가 있어 더 위험하다.

참고값의 출처(지금은 관찰만, 05b에서 증명): $M_n=S_n+aX_n$, $a=\dfrac{2\rho-1}{2-2\rho}=1.5$가 마팅게일이라 $h(i)=\dfrac{i+a}{N+2a}=\dfrac{6.5}{23}=0.282609$. 또는 상태 $(x,d)$의 흡수 사슬(01c의 도구)로 정확히 계산할 수 있다: $h(5)=0.282609$, $E_5[T]=26.25$.

기록할 교훈: 독립성이 깨지면 공식이 틀리지만 **조용히** 틀린다(코드는 에러 없이 돈다). 가정 위반을 잡는 것은 `assert_close`뿐이다.

```python
def simulate_persistent_ruin(i, N, rho, n_paths, rng, max_steps=100_000):
    pos = np.full(n_paths, i, dtype=int)
    last = rng.choice([-1, 1], size=n_paths)                 # X_1 은 대칭
    won = np.zeros(n_paths, dtype=bool); T = np.zeros(n_paths, dtype=int)
    active = np.arange(n_paths)
    for t in range(1, max_steps + 1):
        keep = rng.random(active.size) < rho
        step = np.where(keep, last[active], -last[active])
        last[active] = step
        pos[active] += step
        done = (pos[active] == 0) | (pos[active] == N)
        idx = active[done]; T[idx] = t; won[idx] = pos[idx] == N
        active = active[~done]
        if active.size == 0:
            break
    assert active.size == 0
    return won, T

for i in (5, 10):
    w, T = simulate_persistent_ruin(i, 20, 0.8, N_PATHS, rng)
    print(i, mc_proportion(w), "vs i/N =", i / 20, "| exact (i+1.5)/23 =", (i + 1.5) / 23, "| E[T]", mc_estimate(T))
# i=5:  0.283 ± 0.003 vs 0.25  (i/N 은 4 SE 밖에서 틀린다);  i=10: 0.50 (대칭성 때문에 맞는다)
```

</details>

## 7. 정리

1. 랜덤 워크는 `rng.choice([-1, 1], size=(paths, steps)).cumsum(axis=1)` 한 줄이며, 퍼짐은 $\sqrt n$ 규모다.
2. $(S_n+n)/2\sim\mathrm{Bin}(n,p)$: $S_n$은 $n$과 홀짝이 같은 값만 취한다(원자 간격 2). density 막대 높이 = pmf/칸 폭 이므로, pmf와는 $K_n$으로, 정규 pdf와는 홀수 경계 폭 2 칸으로 비교한다.
3. 꼬리확률은 정규 근사보다 정확한 이항 계산이 기준이고, 연속성 수정($\pm1$)이 오차의 대부분을 없앤다 ($0.0455\to0.0574$ vs 정확값 $0.0569$).
4. 파산 확률은 첫걸음 분석 → 차분방정식 → 기하급수: $p=1/2$이면 $i/N$, 아니면 $(1-(q/p)^i)/(1-(q/p)^N)$; 작은 불리함도 지수적으로 벌을 준다 ($0.5\to0.1185$).
5. 작업 루프: Predict 딕셔너리 → 시뮬레이션 → `compare_table` → `assert_close(k=4)` → mistakes.md. 시뮬레이션은 항상 닫힌 형식과 SE 단위로 대조한다.

**trap 카드**
- Q: $S_{100}$ 히스토그램(`density=True`, `bins=40` on $[-40,40]$)이 이항 pmf의 정확히 절반 높이로 나온다. 난수 생성기 버그인가?
- A: 아니다. $S_{100}$은 짝수만 취하므로(원자 간격 2) 폭 2 칸 하나에 원자가 하나뿐이고 density 높이 = pmf/2 이다(폭 1 칸이면 반대로 정규 pdf의 두 배 빗살). pmf와 비교하려면 $K=(S+n)/2$로, 정규 pdf와 비교하려면 홀수 경계 폭 2 칸으로.

**다음 노트북: 00b** — 이 노트북의 비교표는 'MC 추정 ± 1.96 SE'와 `assert_close(k=4)`를 아무 설명 없이 썼다. 00b에서 SE, 신뢰구간, 오차가 $1/\sqrt N$으로 줄어드는 법칙과 신뢰구간 포함률을 직접 실험해 그 근거를 세운다.

log/progress.md 한 줄 템플릿:
`| YYYY-MM-DD | 00a 랜덤 워크로 시작하기 | __분 | 자기채점 _/5 | 헷갈린 것: ___ (예측 적중 _/6, 다음: 00b) |`